In [1]:

import pandas as pd
from collections import Counter
import numpy as np
import re
import os


df = pd.read_excel("temp1.xlsx")

print(df.head())


   case_id                                  case_title  \
0  CASE_01                   State vs Chaturbhuj Singh   
1  CASE_02  Gurujlal Panika vs State of Madhya Pradesh   
2  CASE_03        Amarsingh vs State of Madhya Pradesh   
3  CASE_04        Nenavath Bujji vs State of Telangana   
4  CASE_05  Amarsingh Badri vs State of Madhya Pradesh   

                          court    year                  case_type  \
0              Delhi High Court  2013.0                   Criminal   
1  high court of madhya pradesh  2023.0                   Criminal   
2        Supreme court of India  1995.0                   Criminal   
3        Supreme court of India  2024.0  Constitutional / Criminal   
4        Supreme court of India  1995.0                   Criminal   

                                   sub_category  \
0  Culpable Homicide / Right of Private Defence   
1           Robbery / Receiving Stolen Property   
2              Murder / Circumstantial Evidence   
3             Preventive D

In [2]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'\n', ' ', text)

    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
df['combined_text'] = (

    df['case_text'].fillna('') + ' ' +

    df['legal_sections'].fillna('') + ' ' +

    df['judge_observation'].fillna('') + ' ' +

    df['legal_keywords'].fillna('') + ' ' +

    df['sub_category'].fillna('')

)

In [5]:
df['combined_text'] = df[
    'combined_text'
].apply(clean_text)

In [6]:


# Check if embeddings already saved
if os.path.exists("case_embeddings.npy"):

    print("Loading saved embeddings...")

    embeddings = np.load(
        "case_embeddings.npy"
    )

else:

    print("Generating embeddings...")

    embeddings = model.encode(
        df['combined_text'].tolist(),
        show_progress_bar=False
    )

    # Save embeddings
    np.save(
        "case_embeddings.npy",
        embeddings
    )

    print("Embeddings saved.")

Loading saved embeddings...


In [7]:
print(len(embeddings))

106


In [8]:
print(len(embeddings[0]))

384


In [9]:
print(df.shape)

(106, 14)


In [10]:

import faiss

# Check if FAISS index exists
if os.path.exists("legal_cases.index"):

    print("Loading saved FAISS index...")

    index = faiss.read_index(
        "legal_cases.index"
    )

else:

    print("Creating FAISS index...")

    index = faiss.IndexFlatL2(
        embeddings.shape[1]
    )

    index.add(
        np.array(embeddings).astype('float32')
    )

    # Save index
    faiss.write_index(
        index,
        "legal_cases.index"
    )

    print("FAISS index saved.")

Loading saved FAISS index...


In [11]:
# Create case type embeddings
case_types = (
    df["case_type"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

case_type_embeddings = model.encode(
    case_types,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
def semantic_case_type_detection(query):

    # Query embedding
    query_embedding = model.encode([query])

    # Similarities
    similarities = cosine_similarity(
        query_embedding,
        case_type_embeddings
    )[0]

    # Best match
    best_idx = similarities.argmax()

    detected_type = case_types[best_idx]

    return detected_type

In [13]:
def smart_retrieval(query, top_k=5):

    # Query embedding
    query_embedding = model.encode([query])

    # Stage 1: global retrieval
    D, I = index.search(
        np.array(query_embedding).astype('float32'),
        k=top_k
    )

    # Detect semantic case type
    dominant_type = semantic_case_type_detection(
        query
    )

    # Filter dataset
    filtered_df = df[
        df["case_type"] == dominant_type
    ].reset_index(drop=True)

    # Create filtered embeddings
    filtered_texts = filtered_df[
        "combined_text"
    ].tolist()

    filtered_embeddings = model.encode(
        filtered_texts,
        show_progress_bar=False
    )

    # Temporary FAISS index
    temp_index = faiss.IndexFlatL2(
        filtered_embeddings.shape[1]
    )

    temp_index.add(
        np.array(filtered_embeddings).astype('float32')
    )

    # Stage 2: refined retrieval
    D2, I2 = temp_index.search(
        np.array(query_embedding).astype('float32'),
        k=min(top_k, len(filtered_df))
    )

    # Similarity filtering
    filtered_indices = []

    filtered_scores = []

    for score, idx in zip(D2[0], I2[0]):

        similarity = 1 / (1 + score)

        if similarity >= 0.45:

            filtered_indices.append(idx)

            filtered_scores.append(similarity)

    # Error handling
    if len(filtered_indices) == 0:

        return None, None, None, None

    return (

        filtered_df,

        filtered_indices,

        filtered_scores,

        dominant_type
    )

In [14]:
query = """
murder using knife and eyewitness
"""

filtered_df, indices, scores, dominant_type = smart_retrieval(query)

print(f"\nDetected Case Type: {dominant_type}\n")

for idx in indices:

    print(f"""
CASE TITLE:
{filtered_df.iloc[idx]['case_title']}

CASE TYPE:
{filtered_df.iloc[idx]['case_type']}

SUB CATEGORY:
{filtered_df.iloc[idx]['sub_category']}

LEGAL SECTIONS:
{filtered_df.iloc[idx]['legal_sections']}

OUTCOME:
{filtered_df.iloc[idx]['judgment_outcome']}

CASE TEXT:
{filtered_df.iloc[idx]['case_text']}

JUDGE OBSERVATION:
{filtered_df.iloc[idx]['judge_observation']}

{"=" * 100}
""")


Detected Case Type: Criminal


CASE TITLE:
Albert Sinha vs State of Assam

CASE TYPE:
Criminal

SUB CATEGORY:
Murder / Assault

LEGAL SECTIONS:
IPC Section 302

OUTCOME:
Guilty

CASE TEXT:
The criminal appeal arose from conviction under murder-related offences. According to the prosecution, the accused allegedly assaulted the victim during a violent altercation resulting in fatal injuries. The prosecution relied upon eyewitness testimony, medical evidence, injury reports, and surrounding circumstances to establish guilt. The defence challenged the consistency of eyewitness accounts and reliability of prosecution evidence. The Court examined whether prosecution successfully proved homicidal assault and criminal intention beyond reasonable doubt.

JUDGE OBSERVATION:
The Court observed that eyewitness testimony corroborated by medical evidence sufficiently established homicidal assault and participation of the accused beyond reasonable doubt.



CASE TITLE:
Jaspreet Singh Sachdeva @ Jass

suggested legal sections

In [15]:
def extract_sections(section_text):

    if pd.isna(section_text):
        return []

    text = str(section_text)

    extracted = []

    # IPC
    ipc_matches = re.findall(
        r'IPC Sections? ([0-9A-Za-z(), ]+)',
        text
    )

    for match in ipc_matches:

        nums = match.split(',')

        for num in nums:

            num = num.strip()

            if num:
                extracted.append(f"IPC {num}")

    # DV Act
    dv_matches = re.findall(
        r'Domestic Violence Act Sections? ([0-9A-Za-z(), ]+)',
        text
    )

    for match in dv_matches:

        nums = match.split(',')

        for num in nums:

            num = num.strip()

            if num:
                extracted.append(f"Domestic Violence Act {num}")

    return extracted

In [16]:
def predict_legal_sections(query, top_k=5):

    # Create embedding
    query_embedding = model.encode([query])

    # Search similar cases
    D, I = index.search(
        np.array(query_embedding).astype('float32'),
        k=top_k
    )

    weighted_sections = {}

    # Loop through retrieved cases
    for position, idx in enumerate(I[0]):

        # Distance score
        distance = D[0][position]

        # Convert distance to weight
        weight = 1 / (1 + distance)

        # Extract sections
        sections = extract_sections(
            df.iloc[idx]['legal_sections']
        )

        # Add weighted score
        for sec in sections:

            if sec not in weighted_sections:
                weighted_sections[sec] = 0

            weighted_sections[sec] += weight

    # Total weighted score
    total_weight = sum(weighted_sections.values())

    results = []

    # Convert to confidence %
    for sec, score in weighted_sections.items():

        confidence = float(round(
            (score / total_weight) * 100,
            2
        ))

        results.append(
            (sec, confidence)
        )

    # Sort descending
    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results

In [17]:
query = """
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.
"""
results = predict_legal_sections(query)



print("\nSuggested Legal Sections:\n")

for sec, confidence in results:

    print(
        f"{sec}  →  {round(confidence,2)}% confidence"
    )


Suggested Legal Sections:

IPC 302  →  49.22% confidence
IPC 302 and related provisions  →  14.49% confidence
IPC 120B  →  12.62% confidence
IPC 449  →  11.84% confidence
IPC 392  →  11.84% confidence


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

missing evidence


In [19]:
from collections import Counter

def detect_missing_evidence(query, top_k=5):

    # Smart retrieval
    filtered_df, indices, scores, dominant_type = smart_retrieval(
        query,
        top_k=top_k
    )

    # Error handling
    if filtered_df is None:

        return {
            "error":
            "No strong matches found"
        }

    all_evidence = []

    # Collect evidence from similar cases
    for idx in indices:

        evidence_text = str(
            filtered_df.iloc[idx]["evidence_types"]
        )

        # Split evidence types
        evidence_list = evidence_text.split(',')

        for ev in evidence_list:

            ev = ev.strip().lower()

            if ev:
                all_evidence.append(ev)

    # Count evidence frequency
    evidence_counts = Counter(all_evidence)

    query_lower = query.lower()

    missing = []

    # Detect potentially missing evidence
    for evidence, count in evidence_counts.items():

        # Common evidence absent in query
        if count >= 2 and evidence not in query_lower:

            missing.append({

                "evidence": evidence
            })

    # Remove duplicates
    unique_missing = []

    seen = set()

    for item in missing:

        if item["evidence"] not in seen:

            unique_missing.append(item)

            seen.add(item["evidence"])

    return unique_missing

In [20]:
query = """
The criminal appeal arose from conviction
of the appellant under murder-related offences.
"""

results = detect_missing_evidence(query)

# Error handling
if isinstance(results, dict):

    print(results["error"])

elif len(results) == 0:

    print("No major missing evidence detected.")

else:

    print("\nPossible Missing Evidence:\n")

    for item in results:

        print(f"- {item['evidence']}")


Possible Missing Evidence:

- eyewitness testimony
- medical evidence
- police investigation records
- postmortem report
- witness testimony
- recovery evidence


evidence to law mappings


In [21]:
import re

def normalize_law(law):

    law = str(law).upper().strip()

    # Remove SECTION word
    law = law.replace(
        "SECTION",
        ""
    )

    # Normalize spaces
    law = re.sub(
        r'\s+',
        ' ',
        law
    )

    # Convert:
    # 302 IPC → IPC 302
    match = re.search(
        r'(\d+)\s*IPC',
        law
    )

    if match:

        number = match.group(1)

        law = f"IPC {number}"

    return law

In [22]:
from collections import defaultdict, Counter

def dataset_evidence_law_mapping(
    query,
    top_k=5
):

    # Smart retrieval
    filtered_df, indices, scores, dominant_type = smart_retrieval(
        query,
        top_k=top_k
    )

    # Error handling
    if filtered_df is None:

        return {
            "error":
            "No strong matches found"
        }    
    

    # Evidence → laws mapping
    evidence_map = defaultdict(list)

    # Process retrieved cases
    for idx in indices:

        evidence_text = str(
            filtered_df.iloc[idx]["evidence_types"]
        )

        legal_text = str(
            filtered_df.iloc[idx]["legal_sections"]
        )

        # Split evidence
        evidence_list = evidence_text.split(',')

        # Split laws
        law_list = legal_text.split(';')

        cleaned_laws = []

        # Clean + normalize law text
        for law in law_list:

            law = law.strip()

            if law and law.lower() != "nan":

                cleaned_laws.append(
                    normalize_law(law)
                )

        # Map evidence → laws
        for ev in evidence_list:

            ev = ev.strip()

            if not ev:
                continue

            evidence_map[ev].extend(
                cleaned_laws
            )

    # Final structured output
    results = []

    for evidence, laws in evidence_map.items():

        # Count law frequency
        law_counts = Counter(laws)

        filtered_laws = []

        # Keep common laws
        for law, count in law_counts.items():

            if count >= 1:

                filtered_laws.append(law)

        # Skip weak mappings
        if len(filtered_laws) == 0:
            continue

        results.append({

            "evidence": evidence,

            "laws": filtered_laws
        })

    return results

In [23]:
print("\nEvidence-to-Law Mapping:\n")

query = """
The criminal appeal arose from conviction
of the appellant under murder-related offences.
"""

results = dataset_evidence_law_mapping(query)

# Error handling
if isinstance(results, dict):

    print(results["error"])

else:

    for item in results:

        # Safety check
        if "laws" not in item:
            continue

        print(f"Evidence: {item['evidence']}")

        print("Related Laws:")

        for law in item["laws"]:

            print(f"   - {law}")

        print("-" * 40)


Evidence-to-Law Mapping:

Evidence: Eyewitness testimony
Related Laws:
   - IPC 302 AND RELATED PROVISIONS
   - IPC 302
   - IPC S 302, 120B
----------------------------------------
Evidence: medical evidence
Related Laws:
   - IPC 302 AND RELATED PROVISIONS
   - IPC 302
   - IPC S 449, 302, 392
   - IPC S 302, 120B
----------------------------------------
Evidence: police investigation records
Related Laws:
   - IPC 302 AND RELATED PROVISIONS
   - IPC 302
   - IPC S 449, 302, 392
   - IPC S 302, 120B
----------------------------------------
Evidence: postmortem report
Related Laws:
   - IPC 302 AND RELATED PROVISIONS
   - IPC 302
   - IPC S 302, 120B
----------------------------------------
Evidence: witness statements
Related Laws:
   - IPC 302 AND RELATED PROVISIONS
----------------------------------------
Evidence: injury reports
Related Laws:
   - IPC 302
----------------------------------------
Evidence: Postmortem report
Related Laws:
   - IPC S 449, 302, 392
------------------

judgement


In [24]:


def judgment_pattern_analytics(
    query,
    top_k=5
):

    # Smart retrieval
    filtered_df, indices, scores, dominant_type = smart_retrieval(
        query,
        top_k=top_k
    )

    # Error handling
    if filtered_df is None:

        return {
            "error":
            "No strong matches found"
        }

    outcome_list = []

    relief_list = []

    # Collect analytics data
    for idx in indices:

        # Judgment outcome
        outcome = str(
            filtered_df.iloc[idx][
                "judgment_outcome"
            ]
        ).strip()

        if outcome and outcome.lower() != "nan":

            outcome_list.append(outcome)

        # Interim relief / custody
        relief = str(
            filtered_df.iloc[idx][
                "interim_relief_or_custody_status"
            ]
        ).strip()

        if relief and relief.lower() != "nan":

            relief_list.append(relief)

    # Count outcomes
    outcome_counts = Counter(outcome_list)

    # Count reliefs
    relief_counts = Counter(relief_list)

    # Total counts
    total_outcomes = sum(
        outcome_counts.values()
    )

    total_reliefs = sum(
        relief_counts.values()
    )

    outcome_results = []

    relief_results = []

    # Outcome percentages
    for outcome, count in outcome_counts.items():

        percentage = round(
            (count / total_outcomes) * 100,
            2
        )

        outcome_results.append({

            "outcome": outcome,

            "percentage": percentage
        })

    # Relief percentages
    for relief, count in relief_counts.items():

        percentage = round(
            (count / total_reliefs) * 100,
            2
        )

        relief_results.append({

            "relief": relief,

            "percentage": percentage
        })

    # Sort descending
    outcome_results.sort(
        key=lambda x: x["percentage"],
        reverse=True
    )

    relief_results.sort(
        key=lambda x: x["percentage"],
        reverse=True
    )

    return {

        "case_type": dominant_type,

        "outcome_analytics": outcome_results,

        "relief_analytics": relief_results
    }

In [25]:
query = """
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.
"""
analytics_results = judgment_pattern_analytics(
    query
)


In [26]:
print("\nJudgment Pattern Analytics:\n")

if "error" in analytics_results:

    print(analytics_results["error"])

else:

    print(
        f"Case Type: {analytics_results['case_type']}"
    )

    # Outcome Trends
    print("\nOutcome Trends:\n")

    for item in analytics_results["outcome_analytics"]:

        print(
            f"- {item['outcome']} "
            f"→ {item['percentage']}%"
        )

    # Relief Trends
    print("\nInterim Relief / Custody Trends:\n")

    for item in analytics_results["relief_analytics"]:

        print(
            f"- {item['relief']} "
            f"→ {item['percentage']}%"
        )


Judgment Pattern Analytics:

Case Type: Criminal

Outcome Trends:

- Guilty → 100.0%

Interim Relief / Custody Trends:

- Accused remained convicted during appeal → 100.0%


In [27]:
def analyze_case(query):



    # Legal section prediction
    legal_sections = predict_legal_sections(
        query
    )

    # Evidence-law mapping
    evidence_mapping = dataset_evidence_law_mapping(
        query
    )

    # Missing evidence detection
    missing_evidence = detect_missing_evidence(
        query
    )

    # Judgment analytics
    analytics = judgment_pattern_analytics(
        query
    )

    # Final combined result
    return {

       

        "legal_sections":
            legal_sections,

        "evidence_mapping":
            evidence_mapping,

        "missing_evidence":
            missing_evidence,

        "analytics":
            analytics
    }

In [28]:
query = """
The accused allegedly attacked
the victim with knife resulting
in death. Eyewitness testimony
and medical evidence were relied upon.
"""

final_results = analyze_case(query)
final_results

{'legal_sections': [('IPC 302', 45.130001068115234),
  ('IPC 120B', 11.270000457763672),
  ('IPC 302 and related provisions', 10.970000267028809),
  ('IPC 449', 10.949999809265137),
  ('IPC 392', 10.949999809265137),
  ('IPC 149', 10.729999542236328)],
 'evidence_mapping': [{'evidence': 'Eyewitness testimony',
   'laws': ['IPC 302',
    'IPC S 302, 120B',
    'IPC 302 AND RELATED PROVISIONS',
    'IPC S 302, 34',
    'IPC S 392, 397, 411, 34']},
  {'evidence': 'medical evidence',
   'laws': ['IPC 302',
    'IPC S 302, 120B',
    'IPC 302 AND RELATED PROVISIONS',
    'IPC S 302, 34']},
  {'evidence': 'postmortem report',
   'laws': ['IPC 302',
    'IPC S 302, 120B',
    'IPC 302 AND RELATED PROVISIONS',
    'IPC S 302, 34']},
  {'evidence': 'injury reports', 'laws': ['IPC 302']},
  {'evidence': 'police investigation records',
   'laws': ['IPC 302',
    'IPC S 302, 120B',
    'IPC 302 AND RELATED PROVISIONS',
    'IPC S 302, 34',
    'IPC S 392, 397, 411, 34']},
  {'evidence': 'recovery 